In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, current_timestamp, hash as spark_hash, pmod, to_date
import json
import os

In [ ]:
BUCKET_COUNT = 64


def load_trip_mapping(databricks_env: str) -> dict:
    with open('./datamapping_Hourly.json', 'r') as mapping_file:
        mapping = json.load(mapping_file)

    if 'Trip' not in mapping:
        raise KeyError('Trip entity not found in datamapping_Hourly.json')

    trip = mapping['Trip']
    source_trip_keys_sql_override = os.environ.get('TRIP_SOURCE_KEY_SQL') or os.environ.get('Trip_Source_Key_SQL')
    return {
        'target_entity': trip['Target_Entity'],
        'target_uc_schema': trip['Target_UC_Schema'],
        'jdbc_url': trip['URL'][databricks_env],
        'driver': trip['Driver'],
        'dbtype': trip['DBType'],
        'keyvault': trip['keyvault'][databricks_env],
        'password_secret': trip['PasswordSecret'][databricks_env],
        'user': trip['NHA'][databricks_env],
        'source_sql': trip['SQL'][databricks_env],
        'source_trip_keys_sql_override': source_trip_keys_sql_override
    }


def ensure_delta_tables(snapshot_table: str, delete_events_table: str):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {snapshot_table} (
        Trip_ID INT NOT NULL,
        Snapshot_Timestamp TIMESTAMP NOT NULL
    )
    USING DELTA
    """)

    # New tables are created bucket-partitioned for scalable delete reconciliation.
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {delete_events_table} (
        Trip_ID INT NOT NULL,
        Trip_ID_Bucket INT NOT NULL,
        Deleted_Detected_AtTimestamp TIMESTAMP NOT NULL,
        Deleted_Detected_Date DATE NOT NULL
    )
    USING DELTA
    PARTITIONED BY (Trip_ID_Bucket)
    """)


def _normalize_sql(sql_text: str) -> str:
    return sql_text.strip().rstrip(';')


def _build_source_trip_keys_sql(trip_config: dict) -> str:
    source_trip_keys_sql_override = trip_config.get('source_trip_keys_sql_override')
    if source_trip_keys_sql_override:
        return f"({_normalize_sql(source_trip_keys_sql_override)}) source_trip_keys"

    return (
        "(SELECT DISTINCT CAST(TRIP_ID AS INT) AS Trip_ID "
        "FROM DRVPAY.TRIP "
        "WHERE TRIP_ID IS NOT NULL) source_trip_keys"
    )


def read_current_source_trip_keys(trip_config: dict) -> DataFrame:
    if trip_config['dbtype'] != 'Oracle':
        raise ValueError(f"Unsupported DB type for Trip delete detection: {trip_config['dbtype']}")

    pwd = dbutils.secrets.get(scope=trip_config['keyvault'], key=trip_config['password_secret'])
    connection_details = {
        'user': trip_config['user'],
        'password': pwd,
        'driver': trip_config['driver']
    }

    source_sql = _build_source_trip_keys_sql(trip_config)
    return spark.read.jdbc(url=trip_config['jdbc_url'], table=source_sql, properties=connection_details).select('Trip_ID')


def get_active_transform_keys(transform_table: str) -> DataFrame:
    transform_df = spark.table(transform_table)
    transform_columns = {field.name.upper() for field in transform_df.schema.fields}

    base_df = transform_df
    if 'IS_DELETED' in transform_columns:
        base_df = base_df.filter((col('IS_DELETED') == False) | col('IS_DELETED').isNull())

    return base_df.select(col('Trip_ID')).where(col('Trip_ID').isNotNull()).dropDuplicates()


def get_historical_raw_keys(raw_table: str) -> DataFrame:
    return spark.table(raw_table).select(col('Trip_ID')).where(col('Trip_ID').isNotNull()).dropDuplicates()


def merge_snapshot(snapshot_table: str, current_source_keys_df: DataFrame):
    snapshot_batch_df = (
        current_source_keys_df
        .withColumn('Snapshot_Timestamp', current_timestamp())
    )

    snapshot_delta = DeltaTable.forName(spark, snapshot_table)
    (
        snapshot_delta.alias('target')
        .merge(
            snapshot_batch_df.alias('source'),
            'target.Trip_ID = source.Trip_ID'
        )
        .whenMatchedUpdate(set={
            'Snapshot_Timestamp': 'source.Snapshot_Timestamp'
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


def detect_deleted_candidates_incremental(
    active_transform_keys_df: DataFrame,
    current_source_keys_df: DataFrame
) -> DataFrame:
    return (
        active_transform_keys_df
        .join(current_source_keys_df, on='Trip_ID', how='left_anti')
        .dropDuplicates(['Trip_ID'])
    )


def detect_deleted_candidates(
    active_transform_keys_df: DataFrame,
    historical_raw_keys_df: DataFrame,
    current_source_keys_df: DataFrame
) -> DataFrame:
    daily_missing_df = active_transform_keys_df.join(
        current_source_keys_df,
        on='Trip_ID',
        how='left_anti'
    )

    historical_missing_df = historical_raw_keys_df.join(
        current_source_keys_df,
        on='Trip_ID',
        how='left_anti'
    )

    return daily_missing_df.unionByName(historical_missing_df).dropDuplicates(['Trip_ID'])


def _trip_bucket(df: DataFrame, trip_col: str = 'Trip_ID') -> DataFrame:
    return df.withColumn('Trip_ID_Bucket', pmod(spark_hash(col(trip_col)), BUCKET_COUNT))


def migrate_delete_events_table_if_needed(delete_events_table: str) -> bool:
    delete_events_df = spark.table(delete_events_table)
    delete_events_columns = {field.name.upper() for field in delete_events_df.schema.fields}
    already_bucketed = ('TRIP_ID_BUCKET' in delete_events_columns and 'DELETED_DETECTED_DATE' in delete_events_columns)
    if already_bucketed:
        return False

    migrated_table = f"{delete_events_table}_bucketed_migration"
    if spark.catalog.tableExists(migrated_table):
        spark.sql(f"DROP TABLE {migrated_table}")

    spark.sql(f"""
    CREATE TABLE {migrated_table}
    USING DELTA
    PARTITIONED BY (Trip_ID_Bucket)
    AS
    SELECT
      CAST(Trip_ID AS INT) AS Trip_ID,
      pmod(hash(CAST(Trip_ID AS INT)), {BUCKET_COUNT}) AS Trip_ID_Bucket,
      COALESCE(Deleted_Detected_AtTimestamp, current_timestamp()) AS Deleted_Detected_AtTimestamp,
      to_date(COALESCE(Deleted_Detected_AtTimestamp, current_timestamp())) AS Deleted_Detected_Date
    FROM {delete_events_table}
    WHERE Trip_ID IS NOT NULL
    """)

    spark.sql(f"DROP TABLE {delete_events_table}")
    spark.sql(f"ALTER TABLE {migrated_table} RENAME TO {delete_events_table}")
    return True


def merge_delete_events(delete_events_table: str, deleted_candidates_df: DataFrame) -> int:
    delete_events_df = spark.table(delete_events_table)
    delete_events_columns = {field.name.upper() for field in delete_events_df.schema.fields}

    candidate_events_df = (
        deleted_candidates_df
        .select('Trip_ID')
        .where(col('Trip_ID').isNotNull())
        .dropDuplicates()
    )

    if 'TRIP_ID_BUCKET' in delete_events_columns:
        candidate_events_df = _trip_bucket(candidate_events_df)
        existing_events_df = delete_events_df.select('Trip_ID', 'Trip_ID_Bucket').dropDuplicates()
        event_batch_df = candidate_events_df.join(
            existing_events_df,
            on=['Trip_ID', 'Trip_ID_Bucket'],
            how='left_anti'
        )

        event_batch_df = (
            event_batch_df
            .withColumn('Deleted_Detected_AtTimestamp', current_timestamp())
            .withColumn('Deleted_Detected_Date', to_date(col('Deleted_Detected_AtTimestamp')))
        )

        delete_events_delta = DeltaTable.forName(spark, delete_events_table)
        (
            delete_events_delta.alias('target')
            .merge(
                event_batch_df.alias('source'),
                'target.Trip_ID = source.Trip_ID AND target.Trip_ID_Bucket = source.Trip_ID_Bucket'
            )
            .whenNotMatchedInsert(values={
                'Trip_ID': 'source.Trip_ID',
                'Trip_ID_Bucket': 'source.Trip_ID_Bucket',
                'Deleted_Detected_AtTimestamp': 'source.Deleted_Detected_AtTimestamp',
                'Deleted_Detected_Date': 'source.Deleted_Detected_Date'
            })
            .execute()
        )
    else:
        # Backward compatibility for pre-partition schema.
        existing_events_df = delete_events_df.select('Trip_ID').dropDuplicates()
        event_batch_df = candidate_events_df.join(existing_events_df, on='Trip_ID', how='left_anti')

        event_batch_df = event_batch_df.withColumn('Deleted_Detected_AtTimestamp', current_timestamp())

        delete_events_delta = DeltaTable.forName(spark, delete_events_table)
        (
            delete_events_delta.alias('target')
            .merge(event_batch_df.alias('source'), 'target.Trip_ID = source.Trip_ID')
            .whenNotMatchedInsert(values={
                'Trip_ID': 'source.Trip_ID',
                'Deleted_Detected_AtTimestamp': 'source.Deleted_Detected_AtTimestamp'
            })
            .execute()
        )

    # Avoid expensive full count() during large backfills; read merge metrics instead.
    history_row = (
        spark.sql(f"DESCRIBE HISTORY {delete_events_table} LIMIT 1")
        .select('operationMetrics')
        .first()
    )
    if history_row and history_row['operationMetrics']:
        return int(history_row['operationMetrics'].get('numTargetRowsInserted', '0'))
    return 0


def create_active_trips_view(active_view: str, transform_table: str, delete_events_table: str):
    delete_event_columns = {field.name.upper() for field in spark.table(delete_events_table).schema.fields}

In [ ]:
def optimize_tables(snapshot_table: str, delete_events_table: str, transform_delete_events_table: str):
    spark.sql(f"OPTIMIZE {snapshot_table}")
    spark.sql(f"OPTIMIZE {delete_events_table} ZORDER BY (Trip_ID)")
    spark.sql(f"OPTIMIZE {transform_delete_events_table} ZORDER BY (Trip_ID)")

In [ ]:
def ensure_transform_delete_events_table(transform_delete_events_table: str):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {transform_delete_events_table} (
        Trip_ID INT NOT NULL,
        Deleted_Detected_AtTimestamp TIMESTAMP NOT NULL,
        Deleted_Detected_Date DATE NOT NULL
    )
    USING DELTA
    """)


def sync_transform_delete_events(delete_events_table: str, transform_delete_events_table: str) -> int:
    deduped_events_df = spark.sql(f"""
    SELECT
        CAST(Trip_ID AS INT) AS Trip_ID,
        MAX(Deleted_Detected_AtTimestamp) AS Deleted_Detected_AtTimestamp,
        TO_DATE(MAX(Deleted_Detected_AtTimestamp)) AS Deleted_Detected_Date
    FROM {delete_events_table}
    WHERE Trip_ID IS NOT NULL
      AND Deleted_Detected_AtTimestamp IS NOT NULL
    GROUP BY Trip_ID
    """)

    deduped_events_count = deduped_events_df.count()

    (
        deduped_events_df.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(transform_delete_events_table)
    )

    return deduped_events_count

In [ ]:
def create_active_trips_view(active_view: str, transform_table: str, delete_events_table: str):
    spark.sql(f"""
    CREATE OR REPLACE VIEW {active_view} AS
    SELECT t.*
    FROM {transform_table} t
    LEFT ANTI JOIN (
        SELECT DISTINCT CAST(Trip_ID AS INT) AS Trip_ID
        FROM {delete_events_table}
        WHERE Trip_ID IS NOT NULL
    ) d
      ON t.Trip_ID = d.Trip_ID
    """)

In [ ]:
# Declare widgets with defaults so the notebook runs standalone or via job.
dbutils.widgets.text('Databricks_Env', '')
dbutils.widgets.text('Loadtracker_Target_Catalog', '')
dbutils.widgets.dropdown('Trip_Delete_Reconcile_Mode', 'incremental', ['incremental', 'full'])
dbutils.widgets.dropdown('Trip_Delete_Events_Migrate', 'false', ['false', 'true'])

def get_widget_or_env(widget_name: str) -> str:
    widget_value = dbutils.widgets.get(widget_name).strip()
    if widget_value:
        return widget_value
    return os.environ.get(widget_name, '').strip()

databricks_env = get_widget_or_env('Databricks_Env')
catalog = get_widget_or_env('Loadtracker_Target_Catalog')
reconcile_mode = dbutils.widgets.get('Trip_Delete_Reconcile_Mode').strip().lower()
enable_delete_table_migration = dbutils.widgets.get('Trip_Delete_Events_Migrate').strip().lower() == 'true'

if not databricks_env:
    raise ValueError("Parameter 'Databricks_Env' is required")
if not catalog:
    raise ValueError("Parameter 'Loadtracker_Target_Catalog' is required")
if reconcile_mode not in ['incremental', 'full']:
    raise ValueError("Parameter 'Trip_Delete_Reconcile_Mode' must be 'incremental' or 'full'")

print(f"[1/8] Starting Trip Delete Detection | env={databricks_env} | catalog={catalog} | mode={reconcile_mode} | migrate={enable_delete_table_migration}")

trip_config = load_trip_mapping(databricks_env)
raw_schema = trip_config['target_uc_schema']
transform_schema = raw_schema.replace('raw', 'transform')
trip_entity = trip_config['target_entity']

raw_trip_table = f"{catalog}.{raw_schema}.{trip_entity}"
transform_trip_table = f"{catalog}.{transform_schema}.{trip_entity}"
snapshot_table = f"{catalog}.{raw_schema}.trip_key_snapshot"
delete_events_table = f"{catalog}.{raw_schema}.trip_delete_events"
transform_delete_events_table = f"{catalog}.{transform_schema}.trip_delete_events"
active_view = f"{catalog}.{transform_schema}.active_trips"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{raw_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{transform_schema}")
ensure_delta_tables(snapshot_table, delete_events_table)
ensure_transform_delete_events_table(transform_delete_events_table)
print(f"[2/8] Schemas and Delta tables ready")

table_migrated = False
if enable_delete_table_migration:
    print(f"[2b/8] Running delete events table migration check...")
    table_migrated = migrate_delete_events_table_if_needed(delete_events_table)
    print(f"[2b/8] Migration complete | migrated={table_migrated}")

print(f"[3/8] Reading current source Trip IDs from Oracle...")
current_source_keys_df = read_current_source_trip_keys(trip_config)
current_source_keys_df.cache()
source_count = current_source_keys_df.count()
print(f"[3/8] Source Trip IDs loaded | count={source_count:,}")

print(f"[4/8] Merging source keys into snapshot table: {snapshot_table}")
merge_snapshot(snapshot_table, current_source_keys_df)
snapshot_total = spark.table(snapshot_table).count()
print(f"[4/8] Snapshot merge complete | total snapshot rows={snapshot_total:,}")

print(f"[5/8] Reading active transform Trip IDs from: {transform_trip_table}")
active_transform_keys_df = get_active_transform_keys(transform_trip_table)
active_transform_keys_df.cache()
active_transform_count = active_transform_keys_df.count()
print(f"[5/8] Active transform Trip IDs loaded | count={active_transform_count:,}")

if reconcile_mode == 'full':
    print(f"[6/8] Full mode: reading historical raw Trip IDs from: {raw_trip_table}")
    historical_raw_keys_df = get_historical_raw_keys(raw_trip_table)
    historical_raw_keys_df.cache()
    historical_raw_count = historical_raw_keys_df.count()
    print(f"[6/8] Historical raw Trip IDs loaded | count={historical_raw_count:,}")
    deleted_candidates_df = detect_deleted_candidates(
        active_transform_keys_df,
        historical_raw_keys_df,
        current_source_keys_df
    )
else:
    print(f"[6/8] Incremental mode: skipping historical raw scan")
    deleted_candidates_df = detect_deleted_candidates_incremental(
        active_transform_keys_df,
        current_source_keys_df
    )

deleted_candidates_df.cache()
candidate_count = deleted_candidates_df.count()
print(f"[6/8] Deleted candidates identified | count={candidate_count:,}")

print(f"[7/8] Merging delete events into raw table: {delete_events_table}")
new_delete_events = merge_delete_events(delete_events_table, deleted_candidates_df)
total_delete_events = spark.table(delete_events_table).count()
transform_delete_events_count = sync_transform_delete_events(delete_events_table, transform_delete_events_table)
print(f"[7/8] Delete events sync complete | raw new this run={new_delete_events:,} | raw total={total_delete_events:,} | transform total={transform_delete_events_count:,}")

print(f"[8/8] Refreshing active trips view and optimizing tables...")
create_active_trips_view(active_view, transform_trip_table, delete_events_table)
optimize_tables(snapshot_table, delete_events_table, transform_delete_events_table)

print()
print("=== Trip Delete Detection Complete ===")
print(f"  Reconcile mode           : {reconcile_mode}")
print(f"  Delete table migrated    : {table_migrated}")
print(f"  Source Trip IDs          : {source_count:,}")
print(f"  Active transform IDs     : {active_transform_count:,}")
print(f"  Deleted candidates       : {candidate_count:,}")
print(f"  New raw delete events    : {new_delete_events:,}")
print(f"  Total raw delete events  : {total_delete_events:,}")
print(f"  Total transform events   : {transform_delete_events_count:,}")
print(f"  Snapshot table           : {snapshot_table}")
print(f"  Raw delete events table  : {delete_events_table}")
print(f"  Transform delete table   : {transform_delete_events_table}")
print(f"  Active view              : {active_view}")